# Xavier / Glorot Init — Understanding the difficulty of training deep feedforward networks (2010)

_Papers · Foundations & Optimization_

**Scale the random starting weights to $\mathrm{Var}(W)=2/(n_\text{in}+n_\text{out})$ so signal and gradient variance stay roughly constant through a deep net instead of vanishing or exploding.**

---

This notebook is a practice scaffold for the **Xavier / Glorot Init — Understanding the difficulty of training deep feedforward networks (2010)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, math
torch.manual_seed(0)

# ---- Xavier uniform from scratch: eq.(16) of Glorot & Bengio (2010) ----
def xavier_uniform(n_in, n_out):
    """W ~ U[-sqrt(6/(n_in+n_out)), +sqrt(6/(n_in+n_out))]  (variance = 2/(n_in+n_out))."""
    limit = math.sqrt(6.0 / (n_in + n_out))            # eq.(16) half-width
    return torch.empty(n_in, n_out).uniform_(-limit, limit)

# ---- THE ORACLE: my bound must equal torch.nn.init.xavier_uniform_'s bound ----
# PyTorch's xavier_uniform_ fills weight (shape out,in) using a = gain*sqrt(6/(fan_in+fan_out)).
n_in, n_out = 256, 128
W = torch.empty(n_out, n_in)                            # PyTorch stores (out_features, in_features)
torch.nn.init.xavier_uniform_(W)                        # gain=1 by default
mine_limit = math.sqrt(6.0 / (n_in + n_out))            # my eq.(16) bound
torch_limit = W.abs().max().item()                      # PyTorch's actual max |weight| ~= its bound
print("my limit  :", round(mine_limit, 6))
print("torch limit:", round(torch_limit, 6))
print("allclose vs torch.nn.init bound:",
      torch.allclose(torch.tensor(mine_limit), torch.tensor(torch_limit), atol=2e-3))  # expect True

# ---- recompute the worked example: n_in=4, n_out=6 ----
nin, nout = 4, 6
varW = 2.0 / (nin + nout)                               # eq.(12) -> 0.2
r = math.sqrt(6.0 / (nin + nout))                       # eq.(16) -> 0.7746
print("worked example: Var(W)=2/(4+6)=", varW, " r=sqrt(6/10)=", round(r, 4),
      " check r^2/3=", round(r*r/3, 4))                 # 0.2, 0.7746, 0.2

# ---- VERIFY: activation + gradient variance through a deep tanh net ----
def deep_tanh_pass(init, depth=10, width=128):
    x = torch.randn(256, width); x = x / x.std()         # unit-variance input
    Ws = []
    for _ in range(depth):
        if init == "xavier":
            Ws.append(xavier_uniform(width, width))      # eq.(16) scaling
        else:                                            # naive "standard" init, eq.(1)
            lim = 1.0 / math.sqrt(width)                 # U[-1/sqrt(n), 1/sqrt(n)] -> n*Var=1/3
            Ws.append(torch.empty(width, width).uniform_(-lim, lim))
    for W in Ws: W.requires_grad_(True)
    h = x; acts = [h.std().item()]
    for W in Ws:
        h = torch.tanh(h @ W); acts.append(h.std().item())
    loss = h.mean()                                      # dummy scalar loss
    loss.backward()
    grads = [W.grad.std().item() for W in Ws]            # per-layer weight-gradient std
    return acts, grads

for init in ["xavier", "naive"]:
    a, g = deep_tanh_pass(init)
    print(f"\n[{init}] activation std by layer:", [round(v, 4) for v in a])
    print(f"[{init}] grad std layer0 vs last : {g[0]:.2e}  ->  {g[-1]:.2e}  (ratio {g[0]/g[-1]:.3f})")
# Xavier: activations stay ~O(0.2), gradient survives to layer 0.
# Naive (standard, n*Var=1/3 < 1): activations collapse toward 0, early gradient ~vanishes.

## Visualize the data & results

_Push a unit-variance input through 10 tanh layers with Xavier vs the paper's 'standard' init — does the activation variance stay stable with Xavier but collapse toward zero with the standard init (the paper's Figure 6)?_

In [ ]:
import numpy as np
def run(init, depth=10, width=128, seed=3):
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((256, width)); x /= x.std()   # unit-variance input
    Ws = []
    for _ in range(depth):
        if init == "normalized":                          # Xavier, eq.(16)
            lim = np.sqrt(6) / np.sqrt(2 * width)
        else:                                             # standard, eq.(1)
            lim = 1.0 / np.sqrt(width)
        Ws.append(rng.uniform(-lim, lim, (width, width)))
    h = x; post = [h]
    for W in Ws:
        h = np.tanh(h @ W); post.append(h)
    act_std = [round(p.std(), 4) for p in post]
    # backprop a unit dummy gradient to read gradient std per layer
    g = rng.standard_normal(post[-1].shape); g /= np.linalg.norm(g)
    gstd = []
    for l in reversed(range(depth)):
        g = g * (1 - post[l + 1] ** 2)                    # tanh'
        gstd.append(g.std()); g = g @ Ws[l].T
    gstd = gstd[::-1]
    return act_std, gstd

for init in ["normalized", "standard"]:
    a, gr = run(init)
    print(init, "act std:", a)
    print(init, "grad ratio layer0/last:", round(gr[0] / gr[-1], 3))
# normalized act std: [1.0, 0.6303, ..., 0.217]   grad ratio ~0.267
# standard   act std: [1.0, 0.4629, ..., 0.0028]   grad ratio ~0.005

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute the Xavier uniform bound for a hidden layer with $n_\text{in}=256$ and $n_\text{out}=256$, and check its variance.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Target variance: $\mathrm{Var}(W)=2/(256+256)=2/512=1/256\approx 0.00391$. — _Eq. (12) with equal widths reduces to $1/n$._
- Half-width: $r=\sqrt{6/512}=\sqrt{0.011719}\approx 0.1083$. — _Eq. (16)._
- Variance of $U[-r,r]$: $r^2/3=(0.1083)^2/3\approx 0.01173/3\approx 0.00391$. — _Confirms it equals the target._

**Answer:** Sample weights from $U[-0.108,\,+0.108]$; the variance is $\approx 0.00391 = 1/256$, exactly eq. (12). For a square layer Xavier's $2/(n_\text{in}+n_\text{out})$ collapses to $1/n$, satisfying both the forward and backward conditions at once.

</details>

**Problem 2.** Why does keeping the per-layer variance factor at 1 stop both vanishing AND exploding signals in a deep net?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- One layer multiplies the variance by $n_\text{in}\,\mathrm{Var}(W)$. — _Sum of independent products (walkthrough step 1)._
- Through $L$ layers the variance is multiplied by that factor $L$ times: $(\text{factor})^L$. — _Each layer applies it again._
- If factor $\lt 1$, $(\text{factor})^L\to 0$ (vanish); if $\gt 1$, $\to\infty$ (explode); if $=1$ it stays put. — _Geometric growth/decay with depth._

**Answer:** Because the factor compounds multiplicatively across depth, anything other than 1 grows or decays geometrically — fatal in deep nets. Setting $n_\text{in}\,\mathrm{Var}(W)=1$ makes the factor exactly 1, so the signal variance is preserved layer after layer. Xavier picks the variance to keep this factor near 1 for both the forward signal and the backward gradient at once.

</details>

**Problem 3.** Ablation: in the CODE's deep-tanh experiment, replace the Xavier init with the naive init (weights $\sim U[-1,1]$, ignoring fan-in). What happens to the activation and gradient variance across the 10 layers, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap the Xavier limit $\sqrt{6/(n_\text{in}+n_\text{out})}$ for a fixed limit of 1. — _Removes the fan-in scaling; per-layer factor is now far from 1._
- Re-run the forward pass and print each layer's activation std. — _Inputs to tanh become large, so tanh saturates to $\pm 1$ (or, for too-small inits, collapse to 0)._
- Re-run the backward pass and print each layer's gradient std. — _Saturated tanh has slope $\approx 0$, so $f'$ kills the back-propagated gradient._

**Answer:** The naive init does not keep the per-layer variance factor near 1, so the variance compounds the wrong way through depth. In our CODEVIZ run, the paper's "standard" init drives the activation std from 1.0 down to ~0.003 by layer 10 (vanishing), and its early-layer gradient is ~200× smaller than its late-layer gradient. Xavier keeps the activation std healthy (1.0 → ~0.22) and the gradient far better preserved (early-layer gradient only ~4× smaller). This is the paper's Figures 6–7 reproduced on toy data: matching the variance budget is what keeps a deep tanh net trainable.

</details>